# Bayesian Dixon–Coles notebook

This notebook implements a simple Bayesian version of a Dixon–Coles-style goal model. Instead of fitting one global model once, it walks through the matches sequentially and refits the model after each new game so that the posterior from the previous step becomes the starting point for the next one.

The implementation uses weakly informative priors, a draw-adjustment term for 0-0 and 1-1 scorelines, and a small, notebook-friendly setup so it can run without requiring a large amount of compute.


In [ ]:
import os
import math
import warnings
import sys
import tqdm
from pathlib import Path


def ensure_project_venv_on_path():
    """Add the repository virtual environment site-packages to sys.path when available."""
    for base in [Path.cwd(), *Path.cwd().parents]:
        candidates = [
            base / ".venv" / "lib" / f"python{sys.version_info.major}.{sys.version_info.minor}" / "site-packages",
            base / ".venv" / "Lib" / "site-packages",
        ]
        for candidate in candidates:
            if candidate.exists() and str(candidate) not in sys.path:
                sys.path.insert(0, str(candidate))
                return str(candidate)
    return None


ensure_project_venv_on_path()

import numpy as np
import pandas as pd
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

# ── Data loading ──────────────────────────────────────────────────────────────
# CSV files matching these patterns in data/ are loaded and concatenated automatically.
# To add new season data, drop a matching file (e.g. finland_5_6_7_divisions_results_26.csv
# or trikiinit_results_26.csv) into the data/ directory and re-run this cell.


def load_matches(data_dir=None):
    """Load and concatenate configured result CSV files found in data_dir."""
    if data_dir is None:
        for base in [Path.cwd(), Path.cwd().parent]:
            candidate = base / "data"
            if candidate.exists():
                data_dir = candidate
                break
    if data_dir is None:
        raise FileNotFoundError("Could not locate the data/ directory.")

    patterns = [
        "finland_5_6_7_divisions_results_*.csv",
        "trikiinit_results_*.csv",
    ]
    csv_files = sorted({p for pattern in patterns for p in Path(data_dir).glob(pattern)})

    if not csv_files:
        raise FileNotFoundError(
            f"No matching CSV files found in {data_dir} for patterns: {patterns}"
        )

    frames = [pd.read_csv(p) for p in csv_files]
    df = pd.concat(frames, ignore_index=True)
    df = df.drop_duplicates(subset=["team_home", "team_away", "goals_home", "goals_away", "season", "group"])
    df = df.loc[df["league_division"].isin([5, 6, 7])].copy()
    df = df.rename(columns={"team_home": "home_team", "team_away": "away_team",
                             "goals_home": "home_goals", "goals_away": "away_goals"})
    df = df.sort_values(["season", "group", "league_division"]).reset_index(drop=True)
    print(f"Loaded {len(df):,} matches (seasons {sorted(df['season'].unique())}) "
          f"from {len(csv_files)} file(s): {[p.name for p in csv_files]}")
    return df


display = lambda value: print(value)
matches = load_matches()
matches.head()

In [ ]:
# Filter to the seasons you want to include in the model.
# Add 26 here once its data lands in the CSV; it will be picked up automatically.
seasons_to_use = [24, 25, 26] #20, 21, 22, 23,

matches = matches[(matches["season"].isin(seasons_to_use))].copy().reset_index(drop=True)

all_teams = sorted(set(matches["home_team"]).union(matches["away_team"]))
team_to_idx = {team: i for i, team in enumerate(all_teams)}

matches["home_idx"] = matches["home_team"].map(team_to_idx)
matches["away_idx"] = matches["away_team"].map(team_to_idx)

display(matches[["home_team", "away_team", "home_goals", "away_goals", "season"]].head())
display(matches.shape)


In [ ]:
fixtures = pd.read_csv("../data/fixtures_remaining_26.csv")
table = pd.read_csv("../data/table_26.csv")

display(fixtures.head())
display(table)

## Model specification

The model uses a Poisson likelihood for home and away goals, with a Dixon–Coles adjustment that slightly increases the probability of draws when both teams score a low number of goals. Team strength is represented by team-specific attack and defense effects, together with a shared home advantage parameter.

To keep the coefficients identifiable, the attack and defense effects are centered so that they sum to zero across teams.

In [ ]:
import pytensor.tensor as pt


def dixon_coles_logp(home_goals, away_goals, home_attack, away_attack, home_defense, away_defense, home_adv, rho, weights=None):
    """Vectorized log-likelihood for a simple Bayesian Dixon–Coles model.

    weights : optional 1-D array of per-match importance weights. When None
              all matches are weighted equally. Use exponential decay weights
              (computed in fit_full_model) to up-weight more recent matches.
    """
    home_goals = pt.flatten(home_goals)
    away_goals = pt.flatten(away_goals)

    home_rate = pt.exp(home_adv + home_attack - away_defense)
    away_rate = pt.exp(away_attack - home_defense)

    logp_home = home_goals * pt.log(home_rate) - home_rate - pt.gammaln(home_goals + 1.0)
    logp_away = away_goals * pt.log(away_rate) - away_rate - pt.gammaln(away_goals + 1.0)

    draw_mask_home = pt.eq(home_goals, 0) & pt.eq(away_goals, 0)
    draw_mask_draw = pt.eq(home_goals, 1) & pt.eq(away_goals, 1)
    draw_correction = pt.where(draw_mask_home, 1.0 - rho, pt.where(draw_mask_draw, 1.0 + rho, 1.0))

    match_logp = logp_home + logp_away + pt.log(draw_correction)
    if weights is not None:
        return pt.sum(weights * match_logp)
    return pt.sum(match_logp)


def fit_full_model(matches, draws=80, tune=80, half_life_seasons=None):
    """Fit the Bayesian Dixon–Coles model once to all selected matches.

    Parameters
    ----------
    half_life_seasons : float or None
        Exponential decay half-life expressed in seasons. A match that is
        `half_life_seasons` seasons older than the newest match receives half
        the likelihood weight of the newest match. Set to None (default) for
        equal weighting.

        Typical choices:
          None  – equal weights (current season as important as season 20)
          3.0   – gentle decay; season 20 gets ~10% weight vs season 25
          2.0   – moderate decay; season 20 gets ~3% weight  (recommended)
          1.0   – aggressive decay; season 20 gets <1% weight
    """
    home_goals = matches["home_goals"].to_numpy(dtype="float64")
    away_goals = matches["away_goals"].to_numpy(dtype="float64")
    home_idx = matches["home_idx"].to_numpy(dtype="int64")
    away_idx = matches["away_idx"].to_numpy(dtype="int64")
    n_matches = len(matches)

    # Compute per-match exponential decay weights (oldest match → lowest weight)
    if half_life_seasons is not None:
        n_seasons = matches["season"].nunique()
        matches_per_season = n_matches / max(n_seasons, 1)
        lam = np.log(2.0) / (half_life_seasons * matches_per_season)
        positions = np.arange(n_matches, dtype="float64")
        raw_weights = np.exp(-lam * (n_matches - 1 - positions))
        # Normalise so the mean weight is 1 (keeps log-likelihood scale stable)
        weights = (raw_weights / raw_weights.mean()).astype("float64")
        weights_tensor = pt.as_tensor_variable(weights)
    else:
        weights_tensor = None

    with pm.Model() as model:
        attack_raw = pm.Normal("attack_raw", mu=0.0, sigma=1.0, shape=len(all_teams))
        defense_raw = pm.Normal("defense_raw", mu=0.0, sigma=1.0, shape=len(all_teams))
        attack = pm.Deterministic("attack", attack_raw - pm.math.mean(attack_raw))
        defense = pm.Deterministic("defense", defense_raw - pm.math.mean(defense_raw))
        home_adv = pm.Normal("home_adv", mu=0.0, sigma=1.0)
        rho = pm.TruncatedNormal("rho", mu=0.05, sigma=0.05, lower=0.0, upper=0.2)

        home_attack = attack[home_idx]
        away_attack = attack[away_idx]
        home_defense = defense[home_idx]
        away_defense = defense[away_idx]

        log_likelihood = dixon_coles_logp(
            home_goals, away_goals,
            home_attack, away_attack,
            home_defense, away_defense,
            home_adv, rho,
            weights=weights_tensor,
        )
        pm.Potential("match_likelihood", log_likelihood)

        trace = pm.sample(
            draws=draws, tune=tune, chains=1, cores=1,
            progressbar=True, target_accept=0.9, random_seed=42, init="auto",
        )

    summary = az.summary(trace.posterior, var_names=["home_adv", "rho", "attack", "defense"], hdi_prob=0.95)
    summary = summary.reset_index().rename(columns={"index": "parameter_name"})

    if "hdi_3%" in summary.columns:
        lower_col, upper_col = "hdi_3%", "hdi_97%"
    elif "hdi_2.5%" in summary.columns:
        lower_col, upper_col = "hdi_2.5%", "hdi_97.5%"
    else:
        lower_col, upper_col = None, None

    summary["parameter"] = summary["parameter_name"].str.extract(r"^(attack|defense|home_adv|rho)")
    summary["team_idx"] = summary["parameter_name"].str.extract(r"\[(\d+)\]")
    summary["team_idx"] = pd.to_numeric(summary["team_idx"], errors="coerce")
    summary["team_name"] = summary["team_idx"].map(lambda i: all_teams[int(i)] if pd.notna(i) else np.nan)

    if lower_col is not None and upper_col is not None:
        summary = summary[
            ["parameter", "parameter_name", "team_name", "team_idx", "mean", "sd", lower_col, upper_col]
        ].rename(columns={lower_col: "hdi_lower", upper_col: "hdi_upper"})
    else:
        summary = summary[["parameter", "parameter_name", "team_name", "team_idx", "mean", "sd"]]
        summary["hdi_lower"] = np.nan
        summary["hdi_upper"] = np.nan

    return trace, summary


def fit_temporal_model(matches, draws=60, tune=60, prior_sigma=0.4, min_matches=5):
    """Fit a joint temporal model with a random-walk prior across seasons."""
    season_ids = matches["season"].astype(int).to_numpy(dtype="int64")
    seasons = np.sort(np.unique(season_ids))
    season_to_idx = {season: i for i, season in enumerate(seasons)}
    season_idx = np.array([season_to_idx[s] for s in season_ids], dtype="int64")

    home_goals = matches["home_goals"].to_numpy(dtype="float64")
    away_goals = matches["away_goals"].to_numpy(dtype="float64")
    home_idx = matches["home_idx"].to_numpy(dtype="int64")
    away_idx = matches["away_idx"].to_numpy(dtype="int64")

    n_seasons = len(seasons)
    n_teams = len(all_teams)

    with pm.Model() as model:
        attack_list = []
        defense_list = []

        for season in range(n_seasons):
            if season == 0:
                attack_prev = 0.0
                defense_prev = 0.0
            else:
                attack_prev = attack_list[-1]
                defense_prev = defense_list[-1]

            attack_s = pm.Normal(f"attack_{season}", mu=attack_prev, sigma=prior_sigma, shape=n_teams)
            defense_s = pm.Normal(f"defense_{season}", mu=defense_prev, sigma=prior_sigma, shape=n_teams)

            attack_list.append(pm.Deterministic(f"attack_centered_{season}", attack_s - pm.math.mean(attack_s)))
            defense_list.append(pm.Deterministic(f"defense_centered_{season}", defense_s - pm.math.mean(defense_s)))

        attack = pt.stack(attack_list)
        defense = pt.stack(defense_list)
        home_adv = pm.Normal("home_adv", mu=0.0, sigma=1.0)
        rho = pm.TruncatedNormal("rho", mu=0.05, sigma=0.05, lower=0.0, upper=0.2)

        home_attack = attack[season_idx, home_idx]
        away_attack = attack[season_idx, away_idx]
        home_defense = defense[season_idx, home_idx]
        away_defense = defense[season_idx, away_idx]

        log_likelihood = dixon_coles_logp(
            home_goals, away_goals,
            home_attack, away_attack,
            home_defense, away_defense,
            home_adv, rho,
        )
        pm.Potential("match_likelihood", log_likelihood)

        trace = pm.sample(
            draws=draws, tune=tune, chains=1, cores=1,
            progressbar=False, target_accept=0.9, random_seed=42, init="auto",
        )

    attack_means = []
    defense_means = []
    for season in range(n_seasons):
        attack_means.append(np.asarray(trace.posterior[f"attack_centered_{season}"].values).mean(axis=(0, 1)))
        defense_means.append(np.asarray(trace.posterior[f"defense_centered_{season}"].values).mean(axis=(0, 1)))

    evolution_rows = []
    for season_idx_value, season in enumerate(seasons):
        for team_idx, team_name in enumerate(all_teams):
            evolution_rows.append({"season": int(season), "team": team_name, "parameter": "attack",  "mean": attack_means[season_idx_value][team_idx]})
            evolution_rows.append({"season": int(season), "team": team_name, "parameter": "defense", "mean": defense_means[season_idx_value][team_idx]})

    evolution_df = pd.DataFrame(evolution_rows)
    return {"trace": trace, "evolution": evolution_df,
            "attack": np.vstack(attack_means), "defense": np.vstack(defense_means), "seasons": seasons}


def plot_temporal_strength_evolution(evolution_df, team_name):
    """Plot how attack and defense estimates change across seasons for a selected team."""
    subset = evolution_df[evolution_df["team"] == team_name].copy()
    plt.figure(figsize=(10, 4))
    for parameter in ["attack", "defense"]:
        parameter_subset = subset[subset["parameter"] == parameter]
        plt.plot(parameter_subset["season"], parameter_subset["mean"], marker="o", label=parameter)

    plt.xticks(sorted(subset["season"].unique()))
    plt.title(f"Temporal strength evolution for {team_name}")
    plt.xlabel("Season")
    plt.ylabel("Posterior mean effect")
    plt.legend()
    plt.tight_layout()
    plt.show()


# ── Within-season evolution ───────────────────────────────────────────────────

def fit_within_season_model(matches, season, n_windows=10, draws=60, tune=60, prior_sigma=0.3, ci=0.68,
                             prior_attack_means=None, prior_defense_means=None, sigma_first_window=None):
    """
    Track how team strengths evolve within a single season.

    Uses an innovation (delta) parameterisation: n_windows independent Normal
    variables are sampled and combined via pt.cumsum to form the random walk.
    This avoids deep recursive Deterministic chains and works for large n_windows.

    Parameters
    ----------
    matches              : DataFrame with 'season', 'home_team', 'away_team',
                           'home_goals', 'away_goals' columns.
    season               : int   – which season to analyse (e.g. 26).
    n_windows            : int   – number of time windows (default 10).
    prior_sigma          : float – within-season window-to-window drift.
    ci                   : float – credible interval width (default 0.68).
    prior_attack_means   : dict  – {team_name: mean} from previous season's last window.
    prior_defense_means  : dict  – same for defense.
    sigma_first_window   : float – uncertainty on the first window (defaults to prior_sigma).
                           Set larger to allow more inter-season change.

    Returns
    -------
    dict with keys: trace, evolution (DataFrame), season_teams, window_labels,
                    team_active_windows, ci.
    """
    season_matches = matches[matches["season"] == season].copy().reset_index(drop=True)
    if len(season_matches) == 0:
        raise ValueError(f"No matches found for season {season}. "
                         "Add the CSV file and re-run the data loading cell.")

    n_matches = len(season_matches)
    matches_per_window = max(1, n_matches // n_windows)
    season_matches["window"] = np.minimum(season_matches.index // matches_per_window, n_windows - 1)

    actual_windows = sorted(season_matches["window"].unique())
    n_windows_actual = len(actual_windows)
    window_to_idx = {w: i for i, w in enumerate(actual_windows)}
    window_idx = season_matches["window"].map(window_to_idx).to_numpy(dtype="int64")

    window_match_bounds: dict = {}
    for row_i, w in enumerate(season_matches["window"]):
        w_idx = window_to_idx[w]
        window_match_bounds.setdefault(w_idx, []).append(row_i + 1)
    window_labels = {w_idx: f"match {min(ms)}–{max(ms)}"
                     for w_idx, ms in window_match_bounds.items()}

    season_teams = sorted(set(season_matches["home_team"]).union(season_matches["away_team"]))
    s_team_to_idx = {t: i for i, t in enumerate(season_teams)}
    home_idx = season_matches["home_team"].map(s_team_to_idx).to_numpy(dtype="int64")
    away_idx = season_matches["away_team"].map(s_team_to_idx).to_numpy(dtype="int64")
    home_goals = season_matches["home_goals"].to_numpy(dtype="float64")
    away_goals = season_matches["away_goals"].to_numpy(dtype="float64")
    n_teams = len(season_teams)

    if sigma_first_window is None:
        sigma_first_window = prior_sigma
    if prior_attack_means is not None:
        first_attack_mu  = np.array([prior_attack_means.get(t, 0.0)  for t in season_teams])
        first_defense_mu = np.array([prior_defense_means.get(t, 0.0) for t in season_teams])
    else:
        first_attack_mu  = np.zeros(n_teams)
        first_defense_mu = np.zeros(n_teams)

    with pm.Model() as model:
        # Innovation parameterisation: sample independent deltas, build random
        # walk via cumsum. This keeps the PyTensor graph shallow regardless of
        # how many windows are used, avoiding the variadic-Add limit.
        delta_attack_0  = pm.Normal("delta_attack_0",  mu=0.0, sigma=sigma_first_window, shape=n_teams)
        delta_defense_0 = pm.Normal("delta_defense_0", mu=0.0, sigma=sigma_first_window, shape=n_teams)

        if n_windows_actual > 1:
            delta_attack_rest  = pm.Normal("delta_attack_rest",  mu=0.0, sigma=prior_sigma, shape=(n_windows_actual - 1, n_teams))
            delta_defense_rest = pm.Normal("delta_defense_rest", mu=0.0, sigma=prior_sigma, shape=(n_windows_actual - 1, n_teams))
            delta_a = pt.concatenate([delta_attack_0[None, :],  delta_attack_rest],  axis=0)
            delta_d = pt.concatenate([delta_defense_0[None, :], delta_defense_rest], axis=0)
        else:
            delta_a = delta_attack_0[None, :]
            delta_d = delta_defense_0[None, :]

        # Random walk: position[w] = prior_mean + cumsum(deltas)[w]
        attack_raw  = first_attack_mu  + pt.cumsum(delta_a, axis=0)   # (n_windows, n_teams)
        defense_raw = first_defense_mu + pt.cumsum(delta_d, axis=0)

        # Center each window so team effects sum to zero
        attack  = pm.Deterministic("attack",  attack_raw  - attack_raw.mean( axis=1, keepdims=True))
        defense = pm.Deterministic("defense", defense_raw - defense_raw.mean(axis=1, keepdims=True))

        home_adv = pm.Normal("home_adv", mu=0.0, sigma=1.0)
        rho = pm.TruncatedNormal("rho", mu=0.05, sigma=0.05, lower=0.0, upper=0.2)

        home_attack  = attack[window_idx,  home_idx]
        away_attack  = attack[window_idx,  away_idx]
        home_defense = defense[window_idx, home_idx]
        away_defense = defense[window_idx, away_idx]

        pm.Potential("match_likelihood", dixon_coles_logp(
            home_goals, away_goals,
            home_attack, away_attack,
            home_defense, away_defense,
            home_adv, rho,
        ))

        trace = pm.sample(
            draws=draws, tune=tune, chains=1, cores=1,
            progressbar=True, target_accept=0.9, random_seed=42, init="auto",
        )

    lo_p = (1.0 - ci) / 2.0 * 100
    hi_p = (1.0 - (1.0 - ci) / 2.0) * 100

    # attack/defense posterior: shape (chain, draw, n_windows, n_teams)
    att_all = np.asarray(trace.posterior["attack"].values).reshape(-1, n_windows_actual, n_teams)
    def_all = np.asarray(trace.posterior["defense"].values).reshape(-1, n_windows_actual, n_teams)

    rows = []
    for w in range(n_windows_actual):
        att_w = att_all[:, w, :]
        def_w = def_all[:, w, :]
        for t_idx, team_name in enumerate(season_teams):
            rows.append({
                "window": w, "label": window_labels[w], "team": team_name,
                "parameter": "attack",
                "mean":  att_w[:, t_idx].mean(),
                "lower": np.percentile(att_w[:, t_idx], lo_p),
                "upper": np.percentile(att_w[:, t_idx], hi_p),
            })
            rows.append({
                "window": w, "label": window_labels[w], "team": team_name,
                "parameter": "defense",
                "mean":  def_w[:, t_idx].mean(),
                "lower": np.percentile(def_w[:, t_idx], lo_p),
                "upper": np.percentile(def_w[:, t_idx], hi_p),
            })

    team_active_windows: dict = {}
    for _, row in season_matches.iterrows():
        w_idx = window_to_idx[row["window"]]
        for team in [row["home_team"], row["away_team"]]:
            team_active_windows.setdefault(team, set()).add(w_idx)

    return {
        "trace": trace,
        "evolution": pd.DataFrame(rows),
        "season_teams": season_teams,
        "window_labels": window_labels,
        "team_active_windows": team_active_windows,
        "ci": ci,
    }


def plot_within_season_evolution(result, team_name, season=None):
    """Plot attack and defense estimates with credible intervals for one team.

    Only windows in which the team actually played a match are shown.
    """
    evo = result["evolution"]
    ci  = result.get("ci", 0.68)
    subset = evo[evo["team"] == team_name].copy()
    if subset.empty:
        raise ValueError(f"'{team_name}' not found. Available: {sorted(evo['team'].unique())[:10]}")

    active = result.get("team_active_windows", {}).get(team_name)
    if active is not None:
        subset = subset[subset["window"].isin(active)]

    windows = sorted(subset["window"].unique())
    labels  = [result["window_labels"][w] for w in windows]
    colors  = {"attack": "C0", "defense": "C1"}

    fig, ax = plt.subplots(figsize=(max(8, len(windows) * 1.4), 4))
    for parameter in ["attack", "defense"]:
        p = subset[subset["parameter"] == parameter].sort_values("window")
        c = colors[parameter]
        ax.plot(p["window"], p["mean"], marker="o", color=c, label=parameter)
        ax.fill_between(p["window"], p["lower"], p["upper"],
                        alpha=0.20, color=c, label=f"{int(ci*100)}% CI ({parameter})")

    ax.set_xticks(windows)
    ax.set_xticklabels(labels, rotation=35, ha="right")
    title = f"Within-season strength evolution — {team_name}"
    if season is not None:
        title += f" (season {season})"
    ax.set_title(title)
    ax.set_xlabel("Match window (season order)")
    ax.set_ylabel("Posterior mean effect")
    ax.legend()
    plt.tight_layout()
    plt.show()


def plot_team_posterior(result, team_name, window=None, ci=0.89):
    """
    Plot the marginal posterior distribution of attack and defense for one team
    at a single time window.

    Parameters
    ----------
    result    : dict returned by fit_within_season_model.
    team_name : str – team to inspect.
    window    : int – local window index (0-based). Defaults to the last window
                where the team played.
    ci        : float – credible interval to shade (default 0.89).
    """
    season_teams = result["season_teams"]
    if team_name not in season_teams:
        raise ValueError(f"'{team_name}' not found. "
                         f"Teams in this result: {sorted(season_teams)[:10]}")
    t_idx = season_teams.index(team_name)

    n_windows = len(result["window_labels"])
    n_teams   = len(season_teams)

    if window is None:
        active = result.get("team_active_windows", {}).get(team_name, set())
        window = max(active) if active else n_windows - 1

    window_label = result["window_labels"].get(window, f"window {window}")

    att_all = np.asarray(result["trace"].posterior["attack"].values).reshape(-1, n_windows, n_teams)
    def_all = np.asarray(result["trace"].posterior["defense"].values).reshape(-1, n_windows, n_teams)
    att_samples = att_all[:, window, t_idx]
    def_samples = def_all[:, window, t_idx]

    lo_p = (1.0 - ci) / 2.0 * 100
    hi_p = (1.0 - (1.0 - ci) / 2.0) * 100

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for ax, samples, param, color in zip(axes,
                                          [att_samples, def_samples],
                                          ["Attack", "Defense"],
                                          ["C0", "C1"]):
        sns.kdeplot(samples, ax=ax, fill=True, color=color, alpha=0.35)
        mean_val = samples.mean()
        lo_val   = np.percentile(samples, lo_p)
        hi_val   = np.percentile(samples, hi_p)
        ax.axvline(mean_val, color="k", linestyle="--", linewidth=1.2,
                   label=f"mean = {mean_val:.3f}")
        ax.axvspan(lo_val, hi_val, alpha=0.12, color=color,
                   label=f"{int(ci*100)}% CI [{lo_val:.2f}, {hi_val:.2f}]")
        ax.set_xlabel(f"{param} coefficient")
        ax.set_ylabel("Density")
        ax.set_title(f"{param}")
        ax.legend(fontsize=8)

    fig.suptitle(f"Posterior marginals — {team_name}  ({window_label})", y=1.02)
    plt.tight_layout()
    plt.show()


# ── Multi-season evolution ────────────────────────────────────────────────────

def fit_all_seasons_model(matches, seasons=None, n_windows=10, draws=60, tune=60, prior_sigma=0.3, ci=0.68):
    """Fit within-season models independently for every season and combine."""
    if seasons is None:
        seasons = sorted(matches["season"].unique())

    all_evolution            = []
    all_window_labels: dict  = {}
    all_team_active_windows: dict = {}
    season_boundaries: dict  = {}
    global_offset            = 0

    for season in seasons:
        if len(matches[matches["season"] == season]) == 0:
            print(f"Season {season}: no data, skipping.")
            continue

        print(f"\n── Season {season} ──")
        result = fit_within_season_model(
            matches, season=season, n_windows=n_windows,
            draws=draws, tune=tune, prior_sigma=prior_sigma, ci=ci,
        )

        n_w = len(result["window_labels"])
        season_boundaries[season] = (global_offset, global_offset + n_w - 1)

        evo = result["evolution"].copy()
        evo["window"] = evo["window"] + global_offset
        evo["season"] = season
        all_evolution.append(evo)

        for w_local, label in result["window_labels"].items():
            all_window_labels[w_local + global_offset] = label

        for team, active_set in result["team_active_windows"].items():
            shifted = {w + global_offset for w in active_set}
            all_team_active_windows.setdefault(team, set()).update(shifted)

        global_offset += n_w

    return {
        "evolution":           pd.concat(all_evolution, ignore_index=True),
        "window_labels":       all_window_labels,
        "team_active_windows": all_team_active_windows,
        "season_boundaries":   season_boundaries,
        "ci":                  ci,
    }


def fit_joint_seasonal_model(matches, seasons=None, n_windows=10, draws=60, tune=60,
                              sigma_within=0.3, sigma_between=0.8, ci=0.68):
    """
    Fit within-season models sequentially, carrying information across seasons.

    Each season's first window is warm-started from the previous season's
    final-window posterior means. `sigma_between` controls how much drift is
    allowed at each season transition: larger values let the model respond
    more freely to inter-season strength changes.

    Parameters
    ----------
    sigma_within  : within-season window-to-window drift.
    sigma_between : uncertainty injected at each season boundary (should be
                    larger than sigma_within to allow inter-season change).
    """
    if seasons is None:
        seasons = sorted(matches["season"].unique())

    all_evolution            = []
    all_window_labels: dict  = {}
    all_team_active_windows: dict = {}
    season_boundaries: dict  = {}
    global_offset            = 0
    prior_attack_means       = None
    prior_defense_means      = None

    for season in seasons:
        if len(matches[matches["season"] == season]) == 0:
            print(f"Season {season}: no data, skipping.")
            continue

        is_first = prior_attack_means is None
        label = "cold start" if is_first else f"warm start (sigma_between={sigma_between})"
        print(f"\n── Season {season} ({label}) ──")

        result = fit_within_season_model(
            matches, season=season, n_windows=n_windows,
            draws=draws, tune=tune,
            prior_sigma=sigma_within, ci=ci,
            prior_attack_means=prior_attack_means,
            prior_defense_means=prior_defense_means,
            sigma_first_window=1.0 if is_first else sigma_between,
        )

        # Extract last-window posterior means to seed the next season
        season_teams = result["season_teams"]
        n_w = len(result["window_labels"])
        att_all = np.asarray(result["trace"].posterior["attack"].values).reshape(-1, n_w, len(season_teams))
        def_all = np.asarray(result["trace"].posterior["defense"].values).reshape(-1, n_w, len(season_teams))
        prior_attack_means  = {t: att_all[:, -1, i].mean() for i, t in enumerate(season_teams)}
        prior_defense_means = {t: def_all[:, -1, i].mean() for i, t in enumerate(season_teams)}

        season_boundaries[season] = (global_offset, global_offset + n_w - 1)

        evo = result["evolution"].copy()
        evo["window"] = evo["window"] + global_offset
        evo["season"] = season
        all_evolution.append(evo)

        for w_local, lbl in result["window_labels"].items():
            all_window_labels[w_local + global_offset] = lbl

        for team, active_set in result["team_active_windows"].items():
            shifted = {w + global_offset for w in active_set}
            all_team_active_windows.setdefault(team, set()).update(shifted)

        global_offset += n_w

    return {
        "evolution":            pd.concat(all_evolution, ignore_index=True),
        "window_labels":        all_window_labels,
        "team_active_windows":  all_team_active_windows,
        "season_boundaries":    season_boundaries,
        "ci":                   ci,
    }


def plot_all_seasons_evolution(result, team_name):
    """
    Plot within- and between-season strength evolution for one team.

    Each plotted point is assigned a sequential integer position so all active
    windows are evenly spaced regardless of gaps in the underlying window index.
    A small gap is inserted between seasons. Season labels sit above the axes
    and the title is raised to avoid overlapping them.
    """
    evo = result["evolution"]
    ci  = result.get("ci", 0.68)
    subset = evo[evo["team"] == team_name].copy()
    if subset.empty:
        raise ValueError(f"'{team_name}' not found. Available: {sorted(evo['team'].unique())[:10]}")

    active = result.get("team_active_windows", {}).get(team_name)
    if active is not None:
        subset = subset[subset["window"].isin(active)]

    if subset.empty:
        print(f"'{team_name}' had no active windows in any season.")
        return

    windows = sorted(subset["window"].unique())

    # Assign sequential plot positions; insert a 1-unit gap between seasons
    global_to_pos: dict = {}
    season_pos_ranges: dict = {}
    pos = 0
    for season, (start, end) in sorted(result["season_boundaries"].items()):
        season_wins = [w for w in windows if start <= w <= end]
        if not season_wins:
            continue
        season_start_pos = pos
        for w in season_wins:
            global_to_pos[w] = pos
            pos += 1
        season_pos_ranges[season] = (season_start_pos, pos - 1)
        pos += 1   # inter-season gap

    colors = {"attack": "C0", "defense": "C1"}
    fig, ax = plt.subplots(figsize=(max(12, len(windows) * 0.9), 4.5))

    for parameter in ["attack", "defense"]:
        p = subset[subset["parameter"] == parameter].sort_values("window")
        positions = [global_to_pos[w] for w in p["window"]]
        c = colors[parameter]
        ax.plot(positions, p["mean"], marker="o", markersize=4, color=c, label=parameter)
        ax.fill_between(positions, p["lower"], p["upper"],
                        alpha=0.20, color=c, label=f"{int(ci*100)}% CI ({parameter})")

    # Season boundary lines and labels
    for i, (season, (spos_start, spos_end)) in enumerate(sorted(season_pos_ranges.items())):
        mid = (spos_start + spos_end) / 2.0
        if i > 0:
            ax.axvline(x=spos_start - 0.5, color="gray", linestyle="--", alpha=0.5, linewidth=1.0)
        ax.text(mid, 1.03, f"Season {season}",
                ha="center", va="bottom", fontsize=8, color="gray",
                transform=ax.get_xaxis_transform())

    all_positions = [global_to_pos[w] for w in windows]
    all_labels    = [result["window_labels"][w] for w in windows]
    step = max(1, len(windows) // 14)
    ax.set_xticks(all_positions[::step])
    ax.set_xticklabels(all_labels[::step], rotation=40, ha="right", fontsize=7)

    ax.set_title(f"Strength evolution across all seasons — {team_name}", pad=28)
    ax.set_xlabel("Match window within season")
    ax.set_ylabel("Posterior mean effect")
    ax.legend(loc="upper left")
    plt.tight_layout(rect=[0, 0, 1, 0.90])
    plt.show()


def plot_full_model_team_posterior(trace, team_name, all_teams, ci=0.89):
    """
    Plot the marginal posterior distribution of attack and defense for one team
    from a fit_full_model trace.

    Parameters
    ----------
    trace     : InferenceData returned by fit_full_model.
    team_name : str   – team to inspect.
    all_teams : list  – the global team list used when the model was fitted.
    ci        : float – credible interval to shade (default 0.89).
    """
    if team_name not in all_teams:
        raise ValueError(f"'{team_name}' not found. Available teams (first 10): {all_teams[:10]}")
    t_idx = all_teams.index(team_name)

    att_samples = np.asarray(trace.posterior["attack"].values).reshape(-1, len(all_teams))[:, t_idx]
    def_samples = np.asarray(trace.posterior["defense"].values).reshape(-1, len(all_teams))[:, t_idx]

    lo_p = (1.0 - ci) / 2.0 * 100
    hi_p = (1.0 - (1.0 - ci) / 2.0) * 100

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for ax, samples, param, color in zip(axes,
                                          [att_samples, def_samples],
                                          ["Attack", "Defense"],
                                          ["C0", "C1"]):
        sns.kdeplot(samples, ax=ax, fill=True, color=color, alpha=0.35)
        mean_val = samples.mean()
        lo_val   = np.percentile(samples, lo_p)
        hi_val   = np.percentile(samples, hi_p)
        ax.axvline(mean_val, color="k", linestyle="--", linewidth=1.2,
                   label=f"mean = {mean_val:.3f}")
        ax.axvspan(lo_val, hi_val, alpha=0.12, color=color,
                   label=f"{int(ci*100)}% CI [{lo_val:.2f}, {hi_val:.2f}]")
        ax.set_xlabel(f"{param} coefficient")
        ax.set_ylabel("Density")
        ax.set_title(param)
        ax.legend(fontsize=8)

    fig.suptitle(f"Posterior marginals (full model) — {team_name}", y=1.02)
    plt.tight_layout()
    plt.show()


## Other functions

In [ ]:
def predict_match_outcomes(home_team, away_team, trace, team_to_idx, all_teams, max_goals=10):
    """Estimate home-win, draw, and away-win probabilities for a matchup from the posterior samples."""
    if home_team not in team_to_idx or away_team not in team_to_idx:
        raise ValueError(f"Unknown team name. Available teams include: {', '.join(all_teams[:10])} ...")

    home_idx = team_to_idx[home_team]
    away_idx = team_to_idx[away_team]

    attack = np.asarray(trace.posterior["attack"].values).reshape(-1, len(all_teams))
    defense = np.asarray(trace.posterior["defense"].values).reshape(-1, len(all_teams))
    home_adv = np.asarray(trace.posterior["home_adv"].values).reshape(-1)
    rho = np.asarray(trace.posterior["rho"].values).reshape(-1)

    goals = np.arange(max_goals + 1)
    home_win_prob = np.zeros(len(home_adv))
    draw_prob = np.zeros(len(home_adv))
    away_win_prob = np.zeros(len(home_adv))

    for sample_idx in range(len(home_adv)):
        lambda_home = np.exp(home_adv[sample_idx] + attack[sample_idx, home_idx] - defense[sample_idx, away_idx])
        lambda_away = np.exp(attack[sample_idx, away_idx] - defense[sample_idx, home_idx])

        home_goal_probs = np.array([
            np.exp(g * np.log(lambda_home) - lambda_home - math.lgamma(g + 1.0)) for g in goals
        ], dtype=float)
        away_goal_probs = np.array([
            np.exp(g * np.log(lambda_away) - lambda_away - math.lgamma(g + 1.0)) for g in goals
        ], dtype=float)

        for home_goals_i, p_home in enumerate(home_goal_probs):
            for away_goals_i, p_away in enumerate(away_goal_probs):
                if home_goals_i == 0 and away_goals_i == 0:
                    correction = 1.0 - rho[sample_idx]
                elif home_goals_i == 1 and away_goals_i == 1:
                    correction = 1.0 + rho[sample_idx]
                else:
                    correction = 1.0

                prob = p_home * p_away * correction
                if home_goals_i > away_goals_i:
                    home_win_prob[sample_idx] += prob
                elif home_goals_i == away_goals_i:
                    draw_prob[sample_idx] += prob
                else:
                    away_win_prob[sample_idx] += prob

    return pd.DataFrame(
        {
            "outcome": ["home_win", "draw", "away_win"],
            "probability": [home_win_prob.mean(), draw_prob.mean(), away_win_prob.mean()],
        }
    ).round(4)

In [ ]:
def plot_match_probability_matrix(home_team, away_team, trace, team_to_idx, all_teams, max_goals=4):
    """Plot a heatmap of posterior probabilities for each possible scoreline outcome."""
    if home_team not in team_to_idx or away_team not in team_to_idx:
        raise ValueError(f"Unknown team name. Available teams include: {', '.join(all_teams[:10])} ...")

    home_idx = team_to_idx[home_team]
    away_idx = team_to_idx[away_team]

    attack = np.asarray(trace.posterior["attack"].values).reshape(-1, len(all_teams))
    defense = np.asarray(trace.posterior["defense"].values).reshape(-1, len(all_teams))
    home_adv = np.asarray(trace.posterior["home_adv"].values).reshape(-1)
    rho = np.asarray(trace.posterior["rho"].values).reshape(-1)

    goals = np.arange(max_goals + 1)
    score_matrix = np.zeros((len(goals), len(goals)))

    for sample_idx in range(len(home_adv)):
        lambda_home = np.exp(home_adv[sample_idx] + attack[sample_idx, home_idx] - defense[sample_idx, away_idx])
        lambda_away = np.exp(attack[sample_idx, away_idx] - defense[sample_idx, home_idx])

        home_goal_probs = np.array([
            np.exp(g * np.log(lambda_home) - lambda_home - math.lgamma(g + 1.0)) for g in goals
        ], dtype=float)
        away_goal_probs = np.array([
            np.exp(g * np.log(lambda_away) - lambda_away - math.lgamma(g + 1.0)) for g in goals
        ], dtype=float)

        for i, p_home in enumerate(home_goal_probs):
            for j, p_away in enumerate(away_goal_probs):
                if i == 0 and j == 0:
                    correction = 1.0 - rho[sample_idx]
                elif i == 1 and j == 1:
                    correction = 1.0 + rho[sample_idx]
                else:
                    correction = 1.0
                score_matrix[i, j] += p_home * p_away * correction

    score_matrix /= len(home_adv)
    score_df = pd.DataFrame(score_matrix, index=[f"{g}" for g in goals], columns=[f"{g}" for g in goals])
    score_df.index.name = "home_goals"
    score_df.columns.name = "away_goals"

    plt.figure(figsize=(7, 6))
    sns.heatmap(score_df, annot=True, fmt=".3f", cmap="Blues", cbar=True)
    plt.title(f"Posterior scoreline probabilities: {home_team} vs {away_team}")
    plt.xlabel("Away goals")
    plt.ylabel("Home goals")
    plt.tight_layout()
    plt.show()

    return score_df

## Full-data fitting

The sequential approach is useful when you want to mimic learning over time, but for estimating the model from the full historical dataset it is usually simpler and more stable to fit once to all matches at once.


In [ ]:
trace, summary = fit_full_model(matches, draws=160, tune=160, half_life_seasons=1.0)
summary.head()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for parameter in ["home_adv", "rho"]:
    subset = summary[summary["parameter"] == parameter]
    ax.bar(parameter, subset.iloc[0]["mean"], yerr=0.0, label=parameter)

ax.set_title("Posterior means for global parameters")
ax.set_xlabel("Parameter")
ax.set_ylabel("Posterior mean")
ax.legend()
plt.tight_layout()
plt.show()

team_posteriors = summary[summary["parameter"].isin(["attack", "defense"])]
team_posteriors = team_posteriors[["parameter", "team_name", "mean", "hdi_lower", "hdi_upper"]].copy()

# HDI width is a simple uncertainty proxy: wider interval -> more posterior uncertainty.
team_posteriors["hdi_width"] = team_posteriors["hdi_upper"] - team_posteriors["hdi_lower"]

team_posteriors = team_posteriors.sort_values(["parameter", "team_name"])
team_posteriors.head(20)


In [ ]:
teams = matches[matches.season==26].home_team.unique()
team_posteriors[team_posteriors['team_name'].isin(teams)].sort_values('hdi_width', ascending=False)

In [ ]:
def goal_scale_rating_uncertainty(trace, all_teams, hdi_prob=0.95):
    """
    Compute uncertainty on goal-based ratings by propagating posterior samples.

    Returns one row per team with mean/HDI/width for:
      - attack_rating (expected goals scored vs median-defense reference)
      - defense_rating (expected goals conceded vs median-attack reference)
      - goal_difference_rating (attack - defense)

    Also includes relative uncertainty ratios:
      - attack_width_over_mean = attack_hdi_width / attack_rating_mean
      - defense_width_over_mean = defense_hdi_width / defense_rating_mean
    """
    n_teams = len(all_teams)

    attack = np.asarray(trace.posterior["attack"].values).reshape(-1, n_teams)
    defense = np.asarray(trace.posterior["defense"].values).reshape(-1, n_teams)
    home_adv = np.asarray(trace.posterior["home_adv"].values).reshape(-1)

    # Use the same style of reference-team anchoring as ratings tables:
    # median attack team and median defense team based on posterior means.
    attack_mean_by_team = attack.mean(axis=0)
    defense_mean_by_team = defense.mean(axis=0)
    med_attack_idx = int(np.argsort(attack_mean_by_team)[n_teams // 2])
    med_defense_idx = int(np.argsort(defense_mean_by_team)[n_teams // 2])

    # Attack rating samples in goals
    att_home = np.exp(home_adv[:, None] + attack - defense[:, [med_defense_idx]])
    att_away = np.exp(attack - defense[:, [med_defense_idx]])
    attack_rating_samples = 0.5 * (att_home + att_away)

    # Defense rating samples in goals conceded
    def_home_conceded = np.exp(attack[:, [med_attack_idx]] - defense)
    def_away_conceded = np.exp(home_adv[:, None] + attack[:, [med_attack_idx]] - defense)
    defense_rating_samples = 0.5 * (def_home_conceded + def_away_conceded)

    goal_diff_samples = attack_rating_samples - defense_rating_samples

    lo_p = (1.0 - hdi_prob) / 2.0 * 100.0
    hi_p = (1.0 - (1.0 - hdi_prob) / 2.0) * 100.0

    rows = []
    for i, team in enumerate(all_teams):
        a = attack_rating_samples[:, i]
        d = defense_rating_samples[:, i]
        g = goal_diff_samples[:, i]

        a_mean = float(a.mean())
        d_mean = float(d.mean())

        a_lo, a_hi = np.percentile(a, [lo_p, hi_p])
        d_lo, d_hi = np.percentile(d, [lo_p, hi_p])
        g_lo, g_hi = np.percentile(g, [lo_p, hi_p])

        a_width = float(a_hi - a_lo)
        d_width = float(d_hi - d_lo)

        rows.append({
            "team": team,
            "attack_rating_mean": a_mean,
            "attack_hdi_lower": float(a_lo),
            "attack_hdi_upper": float(a_hi),
            "attack_hdi_width": a_width,
            "attack_width_over_mean": a_width / a_mean if abs(a_mean) > 1e-12 else np.nan,
            "defense_rating_mean": d_mean,
            "defense_hdi_lower": float(d_lo),
            "defense_hdi_upper": float(d_hi),
            "defense_hdi_width": d_width,
            "defense_width_over_mean": d_width / d_mean if abs(d_mean) > 1e-12 else np.nan,
            "goal_difference_rating_mean": float(g.mean()),
            "goal_diff_hdi_lower": float(g_lo),
            "goal_diff_hdi_upper": float(g_hi),
            "goal_diff_hdi_width": float(g_hi - g_lo),
        })

    out = pd.DataFrame(rows).sort_values("goal_difference_rating_mean", ascending=False).reset_index(drop=True)
    return out

In [ ]:
goal_scale_posteriors = goal_scale_rating_uncertainty(trace, all_teams, hdi_prob=0.95)

# If you want only the current 2026 teams, filter by teams from fixtures.
goal_scale_posteriors_2026 = goal_scale_posteriors[goal_scale_posteriors["team"].isin(teams)].copy()

goal_scale_posteriors_2026

In [ ]:
# Most uncertain teams by goal-difference rating width
# (larger width = more uncertainty, in goals)
goal_scale_posteriors_2026[["team", "goal_difference_rating_mean", "goal_diff_hdi_width"]].sort_values("goal_diff_hdi_width", ascending=False)

In [ ]:
# Marginal posterior from the full-data model — no window dimension, uses global all_teams list.
plot_full_model_team_posterior(trace, "VAMOL/Myyrmäki", all_teams)


In [ ]:
team_posteriors[team_posteriors["parameter"] == "attack"].sort_values("mean", ascending=False).head(10)

In [ ]:
team_posteriors[team_posteriors["parameter"] == "defense"].sort_values("mean", ascending=False).head(10)

In [ ]:
predict_match_outcomes("Trikiinit", "Scorpions", trace, team_to_idx, all_teams)

In [ ]:
score_df = plot_match_probability_matrix("Trikiinit", "Scorpions", trace, team_to_idx, all_teams)

In [ ]:
# Bayesian adapter for Ratings/Simulation (notebook-only, no src edits).
from types import SimpleNamespace


class BayesianPrediction:
    """Small result object compatible with ratings/simulation usage."""
    def __init__(self, grid):
        self.grid = grid
        goals = np.arange(grid.shape[0], dtype=float)
        self.home_goal_expectation = float((grid.sum(axis=1) * goals).sum())
        self.away_goal_expectation = float((grid.sum(axis=0) * goals).sum())


class BayesianPosteriorAdapter:
    """
    Adapter exposing .predict() and .get_params() for existing-style workflow.

    Modes
    -----
    mode='mean'
        Use posterior means for attack/defense/home_adv/rho.
    mode='sample_per_season'
        Draw one posterior sample index per simulated season and keep it fixed
        for all fixtures in that season simulation.
    """
    def __init__(self, trace, all_teams, mode="mean", max_goals=14, random_seed=42):
        if mode not in {"mean", "sample_per_season"}:
            raise ValueError("mode must be 'mean' or 'sample_per_season'")

        self.trace = trace
        self.all_teams = list(all_teams)
        self.team_to_idx = {team: i for i, team in enumerate(self.all_teams)}
        self.mode = mode
        self.max_goals = int(max_goals)
        self.rng = np.random.default_rng(random_seed)

        n_teams = len(self.all_teams)
        self.attack_samples = np.asarray(trace.posterior["attack"].values).reshape(-1, n_teams)
        self.defense_samples = np.asarray(trace.posterior["defense"].values).reshape(-1, n_teams)
        self.home_adv_samples = np.asarray(trace.posterior["home_adv"].values).reshape(-1)
        self.rho_samples = np.asarray(trace.posterior["rho"].values).reshape(-1)
        self.n_samples = len(self.home_adv_samples)

        self.attack_mean = self.attack_samples.mean(axis=0)
        self.defense_mean = self.defense_samples.mean(axis=0)
        self.home_adv_mean = float(self.home_adv_samples.mean())
        self.rho_mean = float(self.rho_samples.mean())

        self._current_sample_idx = None

    def start_simulation(self):
        """Select a single posterior draw for one full simulated season."""
        if self.mode == "sample_per_season":
            self._current_sample_idx = int(self.rng.integers(0, self.n_samples))

    def _get_parameters_for_prediction(self):
        if self.mode == "mean":
            return self.attack_mean, self.defense_mean, self.home_adv_mean, self.rho_mean

        if self._current_sample_idx is None:
            self.start_simulation()

        k = self._current_sample_idx
        return (
            self.attack_samples[k],
            self.defense_samples[k],
            float(self.home_adv_samples[k]),
            float(self.rho_samples[k]),
        )

    def _score_grid(self, lambda_home, lambda_away, rho):
        goals = np.arange(self.max_goals + 1)

        home_pmf = np.array([
            np.exp(g * np.log(lambda_home) - lambda_home - math.lgamma(g + 1.0))
            for g in goals
        ], dtype=float)
        away_pmf = np.array([
            np.exp(g * np.log(lambda_away) - lambda_away - math.lgamma(g + 1.0))
            for g in goals
        ], dtype=float)

        grid = np.outer(home_pmf, away_pmf)
        if self.max_goals >= 1:
            grid[0, 0] *= (1.0 - rho)
            grid[1, 1] *= (1.0 + rho)

        grid_sum = grid.sum()
        if grid_sum <= 0:
            raise ValueError("Invalid probability grid (sum <= 0).")
        return grid / grid_sum

    def predict(self, home_team, away_team):
        if home_team not in self.team_to_idx or away_team not in self.team_to_idx:
            raise ValueError(f"Unknown team pair: {home_team} vs {away_team}")

        attack, defense, home_adv, rho = self._get_parameters_for_prediction()

        h = self.team_to_idx[home_team]
        a = self.team_to_idx[away_team]

        lambda_home = np.exp(home_adv + attack[h] - defense[a])
        lambda_away = np.exp(attack[a] - defense[h])

        grid = self._score_grid(lambda_home, lambda_away, rho)
        return BayesianPrediction(grid)

    def get_params(self):
        # Uses same naming convention as existing workflow (attack_/defence_ keys)
        attack, defense, home_adv, rho = self._get_parameters_for_prediction()
        params = {f"attack_{team}": float(attack[i]) for i, team in enumerate(self.all_teams)}
        params.update({f"defence_{team}": float(defense[i]) for i, team in enumerate(self.all_teams)})
        params["home_advantage"] = float(home_adv)
        params["rho"] = float(rho)
        return params


def get_median_from_list(numbers):
    sorted_numbers = sorted(numbers)
    middle_index = len(sorted_numbers) // 2
    return sorted_numbers[middle_index]


def create_team_ratings_bayesian(clf, teams):
    """Notebook-local copy of ratings logic so no external imports are required."""
    params = clf.get_params()
    attack_params = {k: v for k, v in params.items() if k.startswith("attack_")}
    defense_params = {k: v for k, v in params.items() if k.startswith("defence_")}

    median_attack = get_median_from_list(list(attack_params.values()))
    median_defense = get_median_from_list(list(defense_params.values()))

    median_attack_team = [team.split("attack_")[1] for team, value in attack_params.items() if value == median_attack]
    median_defense_team = [team.split("defence_")[1] for team, value in defense_params.items() if value == median_defense]

    ratings = []
    for team in teams:
        team_attack_rating_home = clf.predict(team, median_defense_team[0]).home_goal_expectation
        team_attack_rating_away = clf.predict(median_defense_team[0], team).away_goal_expectation
        team_attack_rating = np.mean((team_attack_rating_home, team_attack_rating_away))

        team_defense_rating_home = clf.predict(team, median_attack_team[0]).away_goal_expectation
        team_defense_rating_away = clf.predict(median_attack_team[0], team).home_goal_expectation
        team_defense_rating = np.mean((team_defense_rating_home, team_defense_rating_away))

        team_goal_difference_rating = team_attack_rating - team_defense_rating
        ratings.append((team, team_attack_rating, team_defense_rating, team_goal_difference_rating))

    ratings_df = (
        pd.DataFrame(ratings, columns=["team", "attack_rating", "defense_rating", "goal_difference_rating"])
        .sort_values(by="goal_difference_rating", ascending=False)
        .reset_index(drop=True)
    )
    ratings_df.index += 1
    return ratings_df


def _normalize_fixtures(fixtures):
    """Accept either home_team/away_team or Home/Away fixture column naming."""
    out = fixtures.copy()
    if {"home_team", "away_team"}.issubset(out.columns):
        return out
    rename_map = {}
    if "Home" in out.columns:
        rename_map["Home"] = "home_team"
    if "Away" in out.columns:
        rename_map["Away"] = "away_team"
    out = out.rename(columns=rename_map)
    if not {"home_team", "away_team"}.issubset(out.columns):
        raise ValueError("fixtures must include either [home_team, away_team] or [Home, Away].")
    return out


def _simulate_match_with_clf(clf, home_team, away_team, rng):
    probs = clf.predict(home_team, away_team)
    marginal_home = np.sum(probs.grid, axis=1)
    marginal_away = np.sum(probs.grid, axis=0)
    home_goals = int(rng.choice(np.arange(len(marginal_home)), p=marginal_home / marginal_home.sum()))
    away_goals = int(rng.choice(np.arange(len(marginal_away)), p=marginal_away / marginal_away.sum()))

    if home_goals > away_goals:
        outcome = "home_win"
    elif home_goals == away_goals:
        outcome = "draw"
    else:
        outcome = "away_win"

    return outcome, home_goals, away_goals


def _update_league_table(league_table, home_team, away_team, outcome, home_goals, away_goals):
    league_table.loc[league_table["Squad"] == home_team, "MP"] += 1
    league_table.loc[league_table["Squad"] == away_team, "MP"] += 1

    league_table.loc[league_table["Squad"] == home_team, "GF"] += home_goals
    league_table.loc[league_table["Squad"] == home_team, "GA"] += away_goals
    league_table.loc[league_table["Squad"] == home_team, "GD"] += home_goals - away_goals

    league_table.loc[league_table["Squad"] == away_team, "GF"] += away_goals
    league_table.loc[league_table["Squad"] == away_team, "GA"] += home_goals
    league_table.loc[league_table["Squad"] == away_team, "GD"] += away_goals - home_goals

    if outcome == "home_win":
        league_table.loc[league_table["Squad"] == home_team, "Pts"] += 3
        league_table.loc[league_table["Squad"] == home_team, "W"] += 1
        league_table.loc[league_table["Squad"] == away_team, "L"] += 1
    elif outcome == "draw":
        league_table.loc[league_table["Squad"] == home_team, "Pts"] += 1
        league_table.loc[league_table["Squad"] == away_team, "Pts"] += 1
        league_table.loc[league_table["Squad"] == home_team, "D"] += 1
        league_table.loc[league_table["Squad"] == away_team, "D"] += 1
    else:
        league_table.loc[league_table["Squad"] == away_team, "Pts"] += 3
        league_table.loc[league_table["Squad"] == away_team, "W"] += 1
        league_table.loc[league_table["Squad"] == home_team, "L"] += 1


def _simulate_season_with_clf(league_table, fixtures, clf, rng):
    for _, fixture in fixtures.iterrows():
        home_team = fixture.home_team
        away_team = fixture.away_team
        outcome, home_goals, away_goals = _simulate_match_with_clf(clf, home_team, away_team, rng)
        _update_league_table(league_table, home_team, away_team, outcome, home_goals, away_goals)

    league_table = league_table.sort_values(by=["Pts", "GD", "GF"], ascending=False).copy()
    league_table["Rk"] = np.arange(1, league_table.shape[0] + 1)
    return league_table


def simulate_league_bayesian(clf_adapter, league_table, fixtures, nr_simulations, random_seed=123):
    """
    Notebook-local simulation wrapper supporting posterior-draw-per-season mode.
    """
    fixtures_norm = _normalize_fixtures(fixtures)

    required = ["Rk", "Squad", "MP", "W", "D", "L", "GF", "GA", "GD", "Pts"]
    missing = [c for c in required if c not in league_table.columns]
    if missing:
        raise ValueError(f"league_table missing required columns: {missing}")

    rng = np.random.default_rng(random_seed)
    simulation_results = []

    for i in tqdm.tqdm(range(nr_simulations), desc="Simulating..."):
        clf_adapter.start_simulation()
        simulated_table = league_table[required].copy()
        simulated_table = _simulate_season_with_clf(simulated_table, fixtures_norm, clf_adapter, rng)
        simulated_table["simulation_nr"] = i
        simulation_results.append(simulated_table)

    return pd.concat(simulation_results).reset_index(drop=True)


def run_bayesian_ratings_and_simulation(
    trace,
    all_teams,
    teams,
    league_table,
    fixtures,
    nr_simulations=500,
    mode="mean",
    max_goals=14,
    random_seed=42,
):
    """
    Convenience wrapper matching existing Ratings/Simulation workflow.

    mode='mean'               -> deterministic ratings/simulation from posterior means
    mode='sample_per_season'  -> one posterior draw per simulated season
    """
    clf_bayes = BayesianPosteriorAdapter(
        trace=trace,
        all_teams=all_teams,
        mode=mode,
        max_goals=max_goals,
        random_seed=random_seed,
    )

    # Lock one draw while building ratings in sample_per_season mode.
    clf_bayes.start_simulation()
    ratings_df = create_team_ratings_bayesian(clf_bayes, teams)

    simulation_results_df = simulate_league_bayesian(
        clf_adapter=clf_bayes,
        league_table=league_table,
        fixtures=fixtures,
        nr_simulations=nr_simulations,
        random_seed=random_seed + 1,
    )

    return ratings_df, simulation_results_df, clf_bayes

In [ ]:
# Ratings + Simulation with Bayesian full model posterior
# Requires:
#   - trace, all_teams (from fit_full_model)
#   - fixtures and table (e.g. prepared in palloliitto_scrape workflow)

MODE = "sample_per_season"  # "mean" or "sample_per_season"
NR_SIMULATIONS = 200

if "fixtures" not in globals() or "table" not in globals():
    print("Missing `fixtures` and/or `table` in memory. Load/prep them first.")
else:
    # team list can come from fixtures regardless of Home/Away naming
    fixtures_cols = set(fixtures.columns)
    if {"home_team", "away_team"}.issubset(fixtures_cols):
        teams = np.sort(pd.unique(pd.concat([fixtures["home_team"], fixtures["away_team"]], ignore_index=True)))
    elif {"Home", "Away"}.issubset(fixtures_cols):
        teams = np.sort(pd.unique(pd.concat([fixtures["Home"], fixtures["Away"]], ignore_index=True)))
    else:
        raise ValueError("fixtures must include either [home_team, away_team] or [Home, Away].")

    ratings_df, simulation_results_df, bayes_clf = run_bayesian_ratings_and_simulation(
        trace=trace,
        all_teams=all_teams,
        teams=teams,
        league_table=table,
        fixtures=fixtures,
        nr_simulations=NR_SIMULATIONS,
        mode=MODE,
        max_goals=14,
        random_seed=42,
    )

    display(ratings_df.head(15))
    #display(simulation_results_df.head())

In [ ]:
import plotly.graph_objects as go
from plotly.graph_objs import Layout

In [ ]:
my_title = "Miesten Kutonen, Eteläinen 2026"

In [ ]:
my_x = 'attack_rating'

aux = ratings_df.sort_values(by=my_x)

fig = go.Figure()

fig.add_trace(
        go.Bar(
            x=aux[my_x],
            y=aux['team'],#.apply(lambda x: x.split('\\')[0]),
            text=aux[my_x].round(2),
            textposition='outside',
            orientation='h',
            #marker=dict(color = color_list)
        )
)

fig.update_layout(
    width=1000,
    height=800,
    title=f"Attack Ratings - {my_title}<br><sub>Expected goals scored against league average opposition on neutral game venue<sub>",
    template = 'plotly_dark',
    xaxis_title="maalia",
    showlegend=False
    )

fig.show()

In [ ]:
my_x = 'defense_rating'

aux = ratings_df.sort_values(by=my_x, ascending=False if my_x=='defense_rating' else True)

fig = go.Figure()

fig.add_trace(
        go.Bar(
            x=aux[my_x],
            y=aux['team'],#.apply(lambda x: x.split('\\')[0]),
            text=aux[my_x].round(2),
            textposition='outside',
            orientation='h',
            #marker=dict(color = color_list)
        )
)

fig.update_layout(
    width=1000,
    height=800,
    title=f"Defense Ratings - {my_title}<br><sub>Expected goals conceded against league average opposition on neutral game venue<sub>",
    template = 'plotly_dark',
    xaxis_title="maalia",
    showlegend=False
    )

fig.show()

In [ ]:
my_x = 'goal_difference_rating'

aux = ratings_df.sort_values(by=my_x, ascending=False if my_x=='defense_rating' else True)

fig = go.Figure()

fig.add_trace(
        go.Bar(
            x=aux[my_x],
            y=aux['team'],#.apply(lambda x: x.split('\\')[0]),
            text=aux[my_x].round(2),
            textposition='outside',
            orientation='h',
            #marker=dict(color = color_list)
        )
)

fig.update_layout(
    width=1000,
    height=800,
    title=f"Goal Difference Ratings - {my_title}<br><sub>Expected goal difference against league average opposition on neutral game venue<sub>",
    template = 'plotly_dark',
    xaxis_title="maalia",
    showlegend=False
    )

fig.show()

In [ ]:
# Final rank probability matrix from Bayesian simulation results
# Run this after Cell 19 so simulation_results_df exists.

if "simulation_results_df" not in globals():
    print("Run Cell 19 first to create simulation_results_df.")
else:
    nr_sims = simulation_results_df["simulation_nr"].nunique()
    max_rank = int(simulation_results_df["Rk"].max())

    rank_prob_matrix = (
        simulation_results_df.groupby(["Squad", "Rk"]).size()
        .unstack(fill_value=0)
        .reindex(columns=range(1, max_rank + 1), fill_value=0)
    )
    rank_prob_matrix = 100.0 * rank_prob_matrix / nr_sims

    # Reorder teams by mean final rank
    mean_rank_order = simulation_results_df.groupby("Squad")["Rk"].mean().sort_values().index
    rank_prob_matrix = rank_prob_matrix.loc[mean_rank_order]

    plt.figure(figsize=(12, max(5, 0.35 * len(rank_prob_matrix))))
    sns.heatmap(
        rank_prob_matrix,
        annot=True,
        fmt=".1f",
        cmap="Blues",
        linewidths=0.5,
        cbar_kws={"label": "Probability (%)"},
    )
    plt.title(f"Final rank probabilities ({MODE}, n={nr_sims})")
    plt.xlabel("Final league position")
    plt.ylabel("Team")
    plt.tight_layout()
    plt.show()

    #display(rank_prob_matrix.round(2))

## Time-evolution

In [ ]:
temporal_fit = fit_temporal_model(matches, draws=50, tune=50, prior_sigma=0.4)
temporal_fit["evolution"].head()

In [ ]:
plot_temporal_strength_evolution(temporal_fit["evolution"], 'Trikiinit')

## Updating the model with new data

**Adding new season data (e.g. season 26)**

1. Save results to `data/finland_5_6_7_divisions_results_26.csv` with the same columns as the existing file (`team_home`, `team_away`, `goals_home`, `goals_away`, `league_division`, `season`, `group`).
2. Re-run the **data loading cell** — `load_matches()` scans the `data/` folder automatically.
3. Re-run any model cells you want refreshed.

**Updating mid-season as new results come in**

Just append the new rows to your CSV and repeat steps 2–3. The model re-fits from scratch each time using all available data, so posteriors automatically reflect the latest results.

---

## Within-season evolution

`fit_within_season_model` groups a season's rounds into windows (default: 4 rounds each) and fits the same temporal random-walk model within the season. This lets you see whether a team is improving or declining as the season progresses.


In [ ]:
# Within-season evolution example.
# Once season 26 data is loaded, set SEASON = 26 and re-run.
# n_windows controls how many time slices the season is split into.
# Each slice contains roughly len(season_matches) // n_windows consecutive matches.

SEASON = 25        # ← change to 26 when data is available
N_WINDOWS = 10     # ← adjust for coarser/finer resolution

within_season_result = fit_within_season_model(
    matches,
    season=SEASON,
    n_windows=N_WINDOWS,
    draws=60,
    tune=60,
)
within_season_result["evolution"].head(16)


In [ ]:
# Plot within-season evolution for a specific team.
# Replace 'Trikiinit' with any team that plays in the chosen season.
plot_within_season_evolution(within_season_result, "Trikiinit", season=SEASON)


In [ ]:
# Marginal posterior distribution for a team at the final time window.
# Requires a single-season result with a trace (from fit_within_season_model).
# Re-fit if within_season_result is not in memory, or increase draws for a smoother KDE.

within_season_result = fit_within_season_model(
    matches, season=SEASON, n_windows=N_WINDOWS, draws=200, tune=100,
)
plot_team_posterior(within_season_result, "Trikiinit")


## All-seasons evolution

`fit_joint_seasonal_model` fits within-season models sequentially, carrying the previous season's final-window posterior forward as the prior for the next season's first window. `sigma_between` controls how much drift is allowed at each season boundary — set it larger than `sigma_within` so the model can react to genuine inter-season changes without being anchored too tightly to the past.

- **`sigma_within`** — window-to-window drift within a season (smaller → smoother within-season curves).
- **`sigma_between`** — uncertainty injected at each season boundary (larger → model can change more between seasons).


In [ ]:
# Fit all seasons sequentially, each warm-started from the previous season.
# This runs MCMC once per season — expect a few minutes in total.
#
# sigma_within  : within-season drift (smaller = smoother curves within a season)
# sigma_between : between-season drift (larger = model can change more at each new season)

ALL_SEASONS    = [20, 21, 22, 23, 24, 25]   # ← add 26 once data is available
N_WINDOWS      = 100
SIGMA_WITHIN   = 0.3
SIGMA_BETWEEN  = 0.8   # ← tune this to control inter-season responsiveness

joint_result = fit_joint_seasonal_model(
    matches,
    seasons=ALL_SEASONS,
    n_windows=N_WINDOWS,
    sigma_within=SIGMA_WITHIN,
    sigma_between=SIGMA_BETWEEN,
    draws=60,
    tune=60,
)
print("Done. Seasons fitted:", list(joint_result["season_boundaries"].keys()))


In [ ]:
# Plot the full evolution — within-season windows and across-season continuity.
# Season boundaries are marked with dashed lines.
plot_all_seasons_evolution(joint_result, "Trikiinit")


In [ ]:
matches[(matches["season"]==25) & ((matches["home_team"]=="Trikiinit") | (matches["away_team"]=="Trikiinit"))]

In [ ]:
matches.head(50)